# Read HADDOCK ZIP files

In [ ]:
"""
A script to parse and interpret Haddock output files (e.g., file.list, cluster.out).
This script can read plain text files, .zip archives, or .tar.gz archives directly.

If a .zip or .tar.gz archive is provided, it will automatically search for
and parse one of the common Haddock output files (file.list, cluster.out) within it.

This script provides two parsing functions:
1. parse_haddock_output_pandas: (Recommended) Uses the pandas library.
2. parse_haddock_output_native: Uses only built-in Python.

The script is configured to parse the 9 most common columns from a
standard Haddock 2.x/2.4 analysis file.
"""

import sys
import pandas as pd
from pprint import pprint
import zipfile
import io
import tarfile
import os # <-- Added for directory scanning

# Define the standard column names based on Haddock 2.4 output
# This list can be modified if your Haddock version has different outputs.
COLUMN_NAMES = [
    'Structure',
    'HaddockScore',
    'ClusterID',
    'RMSD_from_best',
    'E_elec',
    'E_vdw',
    'E_desolv',
    'E_air',
    'BSA'
]

# Define common Haddock output files to look for inside a zip archive
# It will parse the first one it finds in this list.
COMMON_FILENAMES = ['file.list', 'cluster.out']

def parse_haddock_output_pandas(file_input):
    """
    Parses a Haddock summary file using the pandas library.

    Args:
        file_input (str or file-like object): The path to the Haddock output file
                                            or a file-like object.

    Returns:
        pandas.DataFrame: A DataFrame containing the parsed data,
                          or None if parsing fails.
    """
    print(f"--- Parsing with pandas ---")
    try:
        # Read the file:
        # - delim_whitespace=True: Handles space-separated values.
        # - comment='#': Skips all comment lines.
        # - header=None: Tells pandas the file has no header row.
        df = pd.read_csv(file_input, delim_whitespace=True, comment='#', header=None)

        # Get the number of columns we actually read
        num_read_cols = df.shape[1]

        if num_read_cols == 0:
            print("Error: No data found. The file might be empty or only contain comments.")
            return None

        # Determine how many columns to name
        if num_read_cols < len(COLUMN_NAMES):
            print(f"Warning: File has {num_read_cols} columns, but we expected at least {len(COLUMN_NAMES)}.")
            # Use only as many names as we have columns
            current_cols = COLUMN_NAMES[:num_read_cols]
            df.columns = current_cols
        else:
            # File has enough (or more) columns. Slice to keep only the ones we care about.
            df = df.iloc[:, :len(COLUMN_NAMES)]
            df.columns = COLUMN_NAMES

        print(f"Successfully parsed {len(df)} structures.")
        return df

    except FileNotFoundError:
        print(f"Error: File not found at {filepath}")
        return None
    except pd.errors.EmptyDataError:
        print(f"Error: File is empty or contains no data lines.")
        return None
    except Exception as e:
        print(f"An unexpected error occurred during pandas parsing: {e}")
        return None

def parse_haddock_output_native(file_input):
    """
    Parses a Haddock summary file using only built-in Python.

    Args:
        file_input (str or file-like object): The path to the Haddock output file
                                            or a file-like object.

    Returns:
        list: A list of dictionaries, where each dictionary represents
              a row (structure). Returns None if parsing fails.
    """
    print(f"\n--- Parsing with native Python ---")
    parsed_data = []
    
    # This block handles both file paths (str) and file-like objects
    try:
        if hasattr(file_input, 'read'):
            # It's a file-like object (e.g., from zipfile.open)
            # We wrap it in TextIOWrapper to handle decoding (e.g., UTF-8)
            with io.TextIOWrapper(file_input, encoding='utf-8') as f:
                content = f.readlines()
        else:
            # It's a file path (str)
            with open(file_input, 'r', encoding='utf-8') as f:
                content = f.readlines()

        for line in content:
            # Clean up whitespace
            line = line.strip()

            # Skip empty lines or comment lines
            if not line or line.startswith('#'):
                continue

            # Split the line by whitespace
            parts = line.split()

            row_data = {}
            for i, col_name in enumerate(COLUMN_NAMES):
                # Stop if the line doesn't have enough parts
                if i >= len(parts):
                    break

                try:
                    # Attempt to convert data to a float
                    row_data[col_name] = float(parts[i])
                except ValueError:
                    # If it fails, keep it as a string (e.g., the structure name)
                    row_data[col_name] = parts[i]
            
            if row_data:
                parsed_data.append(row_data)

        print(f"Successfully parsed {len(parsed_data)} structures.")
        return parsed_data

    except FileNotFoundError:
        print(f"Error: File not found at {file_input}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred during native parsing: {e}")
        return None

def process_file(filepath):
    """
    Main processing function.
    Handles either a zip file, a tar.gz file, or a plain text file.
    """
    
    data_df = None
    native_data = None

    if zipfile.is_zipfile(filepath):
        print(f"Input is a zip archive: {filepath}")
        try:
            with zipfile.ZipFile(filepath, 'r') as zf:
                # Get list of files in the zip
                zip_contents = zf.namelist()
                
                # Find the first matching common filename
                file_to_parse = None
                for common_name in COMMON_FILENAMES:
                    # Check for exact match or if the name is at the end of a path
                    found_files = [f for f in zip_contents if f.endswith(common_name)]
                    if found_files:
                        file_to_parse = found_files[0] # Use the first match
                        break
                
                if file_to_parse:
                    print(f"Found '{file_to_parse}' inside the zip.")
                    # Open the file *within* the zip archive
                    with zf.open(file_to_parse) as f:
                        # Pandas can read the file-like object directly
                        data_df = parse_haddock_output_pandas(f)
                    
                    # Re-open for the native parser
                    # (Can't rewind a zip file stream easily, so just re-open)
                    with zf.open(file_to_parse) as f:
                        native_data = parse_haddock_output_native(f)
                else:
                    print(f"Error: Could not find any of {COMMON_FILENAMES} in {filepath}")
                    return

        except zipfile.BadZipFile:
            print(f"Error: File '{filepath}' is not a valid zip file or is corrupted.")
            return
        except Exception as e:
            print(f"An error occurred while processing the zip file: {e}")
            return
            
    elif tarfile.is_tarfile(filepath):
            print(f"Input is a tar archive (tgz/tar.gz): {filepath}")
            try:
                # 'r:*' automatically handles compression (like gzip)
                with tarfile.open(filepath, 'r:*') as tf:
                    tar_contents = tf.getnames()
                    
                    # Find the first matching common filename
                    file_to_parse = None
                    for common_name in COMMON_FILENAMES:
                        # Check for exact match or if the name is at the end of a path
                        found_files = [f for f in tar_contents if f.endswith(common_name)]
                        if found_files:
                            file_to_parse = found_files[0] # Use the first match
                            break
                    
                    if file_to_parse:
                        print(f"Found '{file_to_parse}' inside the tar archive.")
                        
                        # We need to extract the file object twice because
                        # each parser will consume the file stream.
                        f_pandas = tf.extractfile(file_to_parse)
                        f_native = tf.extractfile(file_to_parse)

                        if f_pandas and f_native:
                            # Pass the file-like objects to the parsers
                            data_df = parse_haddock_output_pandas(f_pandas)
                            native_data = parse_haddock_output_native(f_native)
                            
                            # Clean up the file objects
                            f_pandas.close()
                            f_native.close()
                        else:
                            print(f"Error: Could not extract '{file_to_parse}' from tar.")
                            return
                    else:
                        print(f"Error: Could not find any of {COMMON_FILENAMES} in {filepath}")
                        return

            except tarfile.TarError as e:
                print(f"Error: File '{filepath}' is not a valid tar file or is corrupted. Error: {e}")
                return
            except Exception as e:
                print(f"An error occurred while processing the tar file: {e}")
                return
            
    else:
        # It's a plain text file
        print(f"Input is a plain text file: {filepath}")
        data_df = parse_haddock_output_pandas(filepath)
        native_data = parse_haddock_output_native(filepath)

    # --- Process Pandas Results ---
    if data_df is not None:
        print("\nPandas DataFrame Head (Top 5):")
        print(data_df.head())

        # Example analysis: Find the best structure based on HaddockScore
        if 'HaddockScore' in data_df.columns:
            # Sort by HaddockScore (lower is better)
            df_sorted = data_df.sort_values(by='HaddockScore')
            print("\nBest Structure (from pandas):")
            # Use .iloc[0].to_dict() for a nice print
            pprint(df_sorted.iloc[0].to_dict())
        else:
            print("\n'HaddockScore' column not found, can't determine best structure.")


    # --- Process Native Python Results ---
    if native_data:
        print("\nNative Python Data (First 2 entries):")
        pprint(native_data[:2])

        # Example analysis: Find the best structure
        try:
            # Sort by HaddockScore (lower is better)
            native_sorted = sorted(native_data, key=lambda x: x.get('HaddockScore', float('inf')))
            print("\nBest Structure (from native list):")
            pprint(native_sorted[0])
        except Exception as e:
            print(f"Could not sort native data. 'HaddockScore' might be missing or not a number. Error: {e}")

def main():
    """
    Main function to run the parser from the command line.
    The path can be a direct file or a directory to search for an archive.
    """
    input_path = "../Local/HaddockRun"

    if os.path.isdir(input_path):
        print(f"Input is a directory. Searching for archives in: {input_path}")
        # Search for zip or tgz files
        found_file = None
        
        # Sort files to get a consistent order, though not strictly necessary
        try:
            files_in_dir = sorted(os.listdir(input_path))
        except FileNotFoundError:
            print(f"Error: Directory not found at {input_path}")
            sys.exit(1)
        except Exception as e:
            print(f"Error reading directory: {e}")
            sys.exit(1)

        for f in files_in_dir:
            f_path = os.path.join(input_path, f)
            # Check for archive files
            if (f.endswith('.zip') or f.endswith('.tar.gz') or f.endswith('.tgz')) and os.path.isfile(f_path):
                found_file = f_path
                print(f"Found archive: {found_file}")
                break # Process the first one we find
        
        if found_file:
            process_file(found_file)
        else:
            print(f"Error: No .zip, .tar.gz, or .tgz files found in directory {input_path}")
            sys.exit(1)

    elif os.path.isfile(input_path):
        # It's a file, process it directly
        print(f"Input is a file: {input_path}")
        process_file(input_path)
    
    else:
        print(f"Error: Path '{input_path}' is not a valid file or directory.")
        sys.exit(1)


if __name__ == "__main__":
    main()



Input is a directory. Searching for archives in: ../Local/HaddockRun
Found archive: ../Local/HaddockRun\576579-TIMP3_vs_MMP3.tgz
Input is a tar archive (tgz/tar.gz): ../Local/HaddockRun\576579-TIMP3_vs_MMP3.tgz
Found '576579-TIMP3_vs_MMP3/begin/file.list' inside the tar archive.
--- Parsing with pandas ---
Successfully parsed 1000 structures.

--- Parsing with native Python ---
Successfully parsed 1000 structures.

Pandas DataFrame Head (Top 5):
              Structure
0  PREVIT:complex_1.pdb
1  PREVIT:complex_1.pdb
2  PREVIT:complex_1.pdb
3  PREVIT:complex_1.pdb
4  PREVIT:complex_1.pdb

'HaddockScore' column not found, can't determine best structure.

Native Python Data (First 2 entries):
[{'Structure': '"PREVIT:complex_1.pdb"'},
 {'Structure': '"PREVIT:complex_1.pdb"'}]

Best Structure (from native list):
{'Structure': '"PREVIT:complex_1.pdb"'}


C:\Users\ryangustafson\AppData\Local\Temp\ipykernel_30892\3157548974.py:62: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  df = pd.read_csv(file_input, delim_whitespace=True, comment='#', header=None)
